# Imports

In [66]:
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import os

# Model Architecture

In [67]:
feature_extractor = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1), 
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), 
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
)

with torch.no_grad():
    dummy = torch.randn(1, 3, 112, 112)  
    n_features = feature_extractor(dummy).view(1, -1).size(1)

print(f"Flattened feature size: {n_features}")

num_classes = 20
model = nn.Sequential(
    feature_extractor,
    nn.Flatten(),
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, num_classes)
)

Flattened feature size: 25088


# Loading the saved weights

In [68]:
model.load_state_dict(torch.load('epl_logo_cnn.pth',map_location=torch.device('cuda')))
model.eval()

Sequential(
  (0): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=25088, out_features=128, bias=True)
  (3): ReLU()
  (4): Dropout(p=0.5, inplace=False)
  (5): Linear(in_features=128, out_features=20, bias=True)
)

# Preprocess the image

In [111]:
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

image_path = 'external_test_images/brentford.png'
image = Image.open(image_path).convert('RGB')
image = transform(image).unsqueeze(0)

# Run Interence

In [112]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
image = image.to(device)

with  torch.no_grad():
    outputs = model(image)
    probabilities = torch.softmax(outputs,dim=1)
    _,predicted = torch.max(outputs,1)

#  Interpret results

In [113]:
class_names =  ['arsenal', 'aston-villa', 'brentford', 'brighton', 'burnley', 'chelsea', 'crystal-palace', 'everton', 'leeds', 'leicester-city', 'liverpool', 'manchester-city', 'manchester-united', 'newcastle', 'norwich', 'southampton', 'tottenham', 'watford', 'west-ham', 'wolves']

predicted_class = class_names[predicted.item()]
confidence = probabilities[0][predicted.item()].item() * 100 

print(f'Predicted class: {predicted_class}')
print(f'Confidence: {confidence:.2f}%')

print("\nTop 5 probabilities:")
top_probs, top_indices = torch.topk(probabilities[0], 5)
for i in range(5):
    print(f'{class_names[top_indices[i]]}: {top_probs[i] * 100:.2f}%')

Predicted class: brentford
Confidence: 100.00%

Top 5 probabilities:
brentford: 100.00%
chelsea: 0.00%
manchester-united: 0.00%
burnley: 0.00%
southampton: 0.00%
